# OPCal image processing - caiman implementation

### Overview

**TEST PLAN 3 — Deep-learning seeding (CellPose → CNMF).** This notebook is the DL-seeding
variant of the merge-tuning notebook. Instead of CNMF's greedy brightness seeding (which fuses
touching cells), it seeds CNMF with **CellPose-SAM** cell masks via `Ain`, then lets CNMF extract
each cell's activity. CellPose finds *where* the cells are (separating touching pairs); CNMF finds
*what each cell does*. Seeded-CNMF config: `rf=None`, `only_init=False`, `merge_thr=0.95`.

Runs CaImAn CNMF on OPC culture calcium-imaging movies to identify cell ROIs.

### Configuration — edit these paths for your recording
All three paths are relative to the repo. Set your movie, your manual tags, and the CellPose mask you downloaded from Colab (`notebooks/cellpose_colab.ipynb`). See `cnmf_toolkit/DL_SEEDING_WORKFLOW.md`.

In [ ]:
# ==================== CONFIG — edit these 3 paths for your recording ====================
video_path      = r'../data/RawData/20250409_3_Glut_1mM_TIF_VIDEO.tif'        # your movie
annotation_path = r'../data/TaggedData/20250409_3_Glut_1mM_ROI_AllCells.tif'  # your manual tags (used by the compare cell)
# CellPose mask from Colab (cellpose_colab.ipynb), placed in data/results/comparisons/:
#   _seg_labels_corr_d0.npy           = max coverage (recommended default)
#   _seg_labels_corr_d0_ms30_cp1.npy  = stricter / cleaner
MASK_PATH       = r'../data/results/comparisons/_seg_labels_corr_d0.npy'
# =======================================================================================

### Imports and general setup

In [ ]:
# --- Debug-mode toggles (set True to capture per-stage snapshots for cnmf_viewer.py) ---
DEBUG_FIT = True
DEBUG_REFIT = True

if DEBUG_FIT or DEBUG_REFIT:
    import sys
    sys.path.insert(0, '../cnmf_toolkit')
    from instrumented_cnmf import CNMF, debug_tracker
else:
    from caiman.source_extraction.cnmf.cnmf import CNMF
    debug_tracker = None  # not used in non-debug mode

def _set_debug(phase, enabled):
    """Helper: switch the tracker phase + enable/disable. No-op in non-debug mode."""
    if debug_tracker is None:
        return
    debug_tracker.set_phase(phase)
    if enabled:
        debug_tracker.enable()
    else:
        debug_tracker.disable()

# --- Other CaImAn imports preserved from the original cell ---
import bokeh.plotting as bpl
import datetime
import glob
import holoviews as hv
from IPython import get_ipython
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import psutil
from pathlib import Path
import tifffile as tiff
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

try:
    cv2.setNumThreads(0)
except():
    pass

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf, params
from caiman.utils.utils import download_demo
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
from caiman.utils.visualization import nb_view_quilt

bpl.output_notebook()
hv.notebook_extension('bokeh')

# set up logging
logfile = None # Replace with a path if you want to log to a file
logger = logging.getLogger('caiman')
# Set to logging.INFO if you want much output, potentially much more output
logger.setLevel(logging.WARNING)
logfmt = logging.Formatter('%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s')
if logfile is not None:
    handler = logging.FileHandler(logfile)
else:
    handler = logging.StreamHandler()
handler.setFormatter(logfmt)
logger.addHandler(handler)

# set env variables
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

### Visualize data

Generates and plots the maximum intensity projection and local correlation image for an initial inspection of the calcium imaging data.

In [ ]:
movie = cm.load_movie_chain([video_path])

max_projection_orig = np.max(movie, axis=0)
mean_projection = np.mean(movie, axis=0)
correlation_image_orig = cm.local_correlations(movie, swap_dim=False)
correlation_image_orig[np.isnan(correlation_image_orig)] = 0 # get rid of NaNs, if they exist

f, (ax_max, ax_mean, ax_corr) = plt.subplots(1,3,figsize=(12,3))
ax_max.imshow(max_projection_orig,
              cmap='viridis',
              vmin=np.percentile(np.ravel(max_projection_orig),50),
              vmax=np.percentile(np.ravel(max_projection_orig),99.5))
ax_max.set_title("Max Projection Orig", fontsize=12)

ax_mean.imshow(mean_projection,
              cmap='viridis',
              vmin=np.percentile(np.ravel(mean_projection),50),
              vmax=np.percentile(np.ravel(mean_projection),99.5))
ax_mean.set_title("Mean Projection Orig", fontsize=12)

ax_corr.imshow(correlation_image_orig,
               cmap='viridis',
               vmin=np.percentile(np.ravel(correlation_image_orig),50),
               vmax=np.percentile(np.ravel(correlation_image_orig),99.5))
ax_corr.set_title('Correlation Image Orig', fontsize=12)

#save mean projection for later
vmin = np.percentile(mean_projection, 50)
vmax = np.percentile(mean_projection, 99.5)
mean_proj_norm = np.clip((mean_projection - vmin) / (vmax - vmin), 0, 1)

#File preview
plt.figure(figsize=(6, 6))
plt.imshow(mean_proj_norm, cmap='gray')
plt.title("File preview (mean projection)", fontsize=14)
plt.axis('off')
plt.show()

### Set initial parameters

In [ ]:
### CNMF parameters — TEST PLAN 3 (DL-seeding, FROZEN seeded CNMF) ###
# CellPose (Ain, built in the next cell) provides the cells; CNMF only extracts their traces with
# the footprints FROZEN. So the detection params below don't drive detection here — the ones that
# matter are rf=None (no greedy patches) and only_init=False (seeded init), plus imaging metadata.
# https://caiman.readthedocs.io/en/latest/Getting_Started.html

# imaging metadata
fr = 1.08                     # frames per second                                ### User input ###
decay_time = 20               # length of a typical transient (seconds)
dxy = (1.243, 1.243)          # spatial resolution, um per pixel                 ### User input ###

# CNMF params
p = 0                         # AR order (0 = no deconvolution)
gnb = 2                       # number of global background components
merge_thr = 0.95              # kept in params; the frozen path runs no merge step
bas_nonneg = True             # nonnegativity on the trace baseline
rf = None                     # SEEDED CNMF requires rf=None (no patches; seeds come from the CellPose Ain)
gSig = np.array([3, 3])       # cell half-size (px), used by the temporal model
gSiz = 2*gSig + 1             # Gaussian kernel width/height
method_init = 'greedy_roi'    # unused when Ain is provided (seeds override init)
ssub = 1                      # spatial subsampling
tsub = 1                      # temporal subsampling

# parameters dictionary
parameter_dict = {'fnames': video_path,
                  'fr': fr,
                  'dxy': dxy,
                  'decay_time': decay_time,
                  'p': p,
                  'nb': gnb,
                  'rf': rf,
                  'gSig': gSig,
                  'gSiz': gSiz,
                  'method_init': method_init,
                  'rolling_sum': True,
                  'only_init': False,          # SEEDED CNMF: run the temporal update after seeding from Ain (must be False)
                  'ssub': ssub,
                  'tsub': tsub,
                  'merge_thr': merge_thr,
                  'bas_nonneg': bas_nonneg,
                  'use_cnn': False,
                  }

parameters = params.CNMFParams(params_dict=parameter_dict) # CNMFParams is the parameters class

### Set up multicore processing

Automatically configures parallel processing using available CPU cores to speed up CNMF fitting.

In [ ]:
print(f"You have {psutil.cpu_count()} CPUs available in your current environment")
num_processors_to_use = None

if 'cluster' in locals():  # 'locals' contains list of current local variables
    print('Closing previous cluster')
    cm.stop_server(dview=cluster)
print("Setting up new cluster")
_, cluster, n_processes = cm.cluster.setup_cluster(backend='multiprocessing',
                                                   n_processes=num_processors_to_use,
                                                   ignore_preexisting=False)
print(f"Successfully initilialized multicore processing with a pool of {n_processes} CPU cores")

### Creating and accessing memory mapped files

Converts the TIFF video to a memory-mapped format for efficient reading.

In [ ]:
border_to_0 = 0 # number of pixels to exclude from the boundaries

# Motion correction toggle. exp10 showed rigid MC made detection WORSE (lost cells, junk unchanged)
# -> OFF. The streak/background artifacts are not motion.
MOTION_CORRECT = False   # exp10 = ON (failed); keep OFF for the greedy baseline

if MOTION_CORRECT:
    from caiman.motion_correction import MotionCorrect
    mc = MotionCorrect([video_path], dview=cluster, max_shifts=(5, 5),
                       pw_rigid=False, border_nan='copy')
    mc.motion_correct(save_movie=True)
    mc_memmapped_fname = cm.save_memmap(mc.mmap_file, base_name='memmap_mc_',
                                        order='C', border_to_0=0, dview=cluster)
    print('Motion correction ON — memmap built from the corrected movie')
else:
    mc_memmapped_fname = cm.save_memmap([video_path], base_name='memmap_',
                                        order='C', border_to_0=0, dview=cluster)
    print('Motion correction OFF — memmap built from the raw movie')

In [ ]:
Yr, dims, num_frames = cm.load_memmap(mc_memmapped_fname)
images = np.reshape(Yr.T, [num_frames] + list(dims), order='F') #reshape frames in standard 3d format (T x X x Y)



In [ ]:
cm.stop_server(dview=cluster)
_, cluster, n_processes = cm.cluster.setup_cluster(backend='multiprocessing',
                                                   n_processes=num_processors_to_use,
                                                   single_thread=False)

### Build CellPose seeds → `Ain`  (TEST PLAN 3)

Load the CellPose-SAM **D0 (native) label mask** produced on Colab (`_seg_labels_corr_d0.npy`,
374 cells) and convert it into CaImAn's seeded-initialization matrix `Ain` — one sparse column
per cell, pixels **Fortran-flattened** (the order CaImAn/`ground_truth_scorer.py` use). The next
cell passes this `Ain` to the CNMF constructor so CellPose's cells become CNMF's initial
components instead of greedy brightness seeds.

To try a different segmentation (d4/d6/d8, or the mean/max projection), just change `MASK_PATH`.

In [ ]:
# === Build Ain (CellPose seeds) + bootstrap Cin/b_in/f_in for seeded CNMF ===
# CaImAn 1.13.1's fit() SKIPS initialization when A is provided, then calls update_spatial,
# which requires C and f. So for seeded CNMF we must supply the temporal (Cin) and background
# (b_in, f_in) too — bootstrapped here from the seeds + the movie. CNMF then refines them.
# MASK_PATH is set in the CONFIG cell at the top of the notebook.
import scipy.sparse
from scipy.sparse.linalg import svds

# INTENSITY_WEIGHT: build soft footprints (mask x brightness) instead of flat binary masks.
# Soft footprints hug the bright cell core -> tighter ROIs in the overlay + fewer edge-clip
# merges. This is the winning default. False = the original hard binary masks.
INTENSITY_WEIGHT = True

_lbl = np.load(MASK_PATH)
_n_seeds = int(_lbl.max())
_H, _W = _lbl.shape
assert _H * _W == int(np.prod(dims)), (
    f'mask {_lbl.shape} does not match movie dims {dims} — re-export the projection for this recording')

# --- Ain: one column per cell, Fortran-flattened (matches CaImAn spatial A and Yr's pixel order) ---
_wimg = np.nan_to_num(correlation_image_orig, nan=0.0).astype('float32')   # brightness = the corr image CellPose used
_cols = []
for _j in range(1, _n_seeds + 1):
    _m = (_lbl == _j)
    if INTENSITY_WEIGHT:
        _w = np.maximum(_m.astype('float32') * _wimg, 0.0)     # soft, non-negative
        _mx = _w.max()
        _col = (_w / _mx) if _mx > 1e-9 else _m.astype('float32')   # peak-normalized; fallback to binary
    else:
        _col = _m.astype('float32')                            # hard binary mask
    _cols.append(_col.flatten(order='F'))
Ain = scipy.sparse.csc_matrix(np.array(_cols).T)          # (npix, n_seeds)

# --- Cin: mean fluorescence trace inside each seed mask (npix-normalized) ---
# Yr is (npix, T) from the memmap, same Fortran pixel order as Ain.
_colsum = np.asarray(Ain.sum(0)).ravel() + 1e-9           # (n_seeds,)
Cin = np.asarray(Ain.T @ Yr, dtype='float32')             # (n_seeds, T)
Cin = np.maximum(Cin / _colsum[:, None], 0.0)

# --- b_in / f_in: rank-gnb background from the movie's dominant low-rank structure ---
# Use abs (svd components are signed) + a mean fallback so NO background component is all-zero,
# which would crash update_temporal's empty-component pruning in the FREEZE (temporal-only) path.
_U, _S, _Vt = svds(np.asarray(Yr, dtype='float32'), k=gnb)
b_in = np.abs(_U * _S).astype('float32')                   # (npix, gnb)
f_in = np.abs(_Vt).astype('float32')                       # (gnb, T)
_mT = np.abs(np.asarray(Yr.mean(0)).ravel()).astype('float32')   # (T,)
_mP = np.abs(np.asarray(Yr.mean(1)).ravel()).astype('float32')   # (npix,)
for _k in range(gnb):
    if f_in[_k].sum() < 1e-6: f_in[_k] = _mT
    if b_in[:, _k].sum() < 1e-6: b_in[:, _k] = _mP

print(f'Loaded CellPose mask: {MASK_PATH}  (INTENSITY_WEIGHT={INTENSITY_WEIGHT})')
print(f'  seeds={_n_seeds}  Ain {Ain.shape} (nnz={Ain.nnz})')
print(f'  Cin {Cin.shape},  b_in {b_in.shape},  f_in {f_in.shape}')
print('  Pass Ain/Cin/b_in/f_in to CNMF(...) in the next cell for seeded initialization.')

### (Optional) Junk pre-filter — drop dim/moving seeds · OFF by default
Set `JUNK_FILTER = 'min'` (recommended) or `'mean'` to drop the dimmest seeds before CNMF — targets dead/moving artifacts. `'min'` is the strong one (~35 junk / ~3 real cells at 10%); `'mean'` is expected weak.

In [ ]:
# (Optional) JUNK PRE-FILTER — drop the dimmest/most-transient seed footprints before CNMF.
# Idea (Lotan): dead/moving cells smear -> dim on a time-projection. The MIN projection is bright
# only where something is present in EVERY frame, so moving objects go dark there.
# Diagnostic on the d0 mask: min-projection bottom-10% dropped ~35 junk for only ~3 real cells
# (best junk trade found). 'mean' expected WEAKER (junk vs real medians were ~equal).
JUNK_FILTER = 'min'       # None (off) | 'min' | 'mean'
JUNK_DROP_PCT = 10        # drop the dimmest this-percent of seeds
if JUNK_FILTER is not None:
    _proj = movie.min(0) if JUNK_FILTER == 'min' else movie.mean(0)
    _lo, _hi = np.percentile(_proj, (1, 99.9))
    _projn = np.clip((_proj - _lo) / (_hi - _lo + 1e-9), 0, 1)
    _bright = np.array([_projn[_lbl == j].mean() for j in range(1, _n_seeds + 1)])
    _keep = _bright >= np.percentile(_bright, JUNK_DROP_PCT)
    _n0 = Ain.shape[1]
    Ain = Ain[:, _keep]
    Cin = Cin[_keep]
    print(f"JUNK_FILTER='{JUNK_FILTER}' (drop dimmest {JUNK_DROP_PCT}%): seeds {_n0} -> {Ain.shape[1]} "
          f"(dropped {_n0 - Ain.shape[1]})")
else:
    print('JUNK pre-filter OFF — keeping all CellPose seeds.')

### Seeded CNMF — hand CellPose's cells to CNMF

In [ ]:
### Construct the seeded CNMF: hand it the CellPose seeds (Ain + Cin + b_in + f_in).
### With rf=None the cells come from CellPose, not greedy patches; the next cell freezes the
### footprints and only extracts each cell's trace.
cnmf_model = CNMF(n_processes,
                  params=parameters,
                  dview=cluster,
                  Ain=Ain, Cin=Cin, b_in=b_in, f_in=f_in)
print(f'Seeded CNMF ready: {Ain.shape[1]} CellPose seeds wired in (rf=None -> global seeded processing, no patches).')

#### Extract traces from the frozen CellPose seeds

In [ ]:
%%time
# Seeded CNMF — FREEZE FOOTPRINTS: keep A = the CellPose masks FIXED and only extract each
# cell's trace (temporal update). No spatial update (which would grow footprints back into
# merges) and no refit. This is the winning hybrid: CellPose separates the cells, CNMF reads
# their activity, and the separation is preserved exactly.
_set_debug('fit', DEBUG_FIT)
_T = images.shape[0]
cnmf_model.dims = images.shape[1:]
cnmf_model.params.set('online', {'init_batch': _T})
# per-process pixel-chunk setup that fit() normally does before preprocess
if cnmf_model.params.get('preprocess', 'n_pixels_per_process') is None:
    _avail = psutil.virtual_memory()[1] / 2.**30 / n_processes
    _npx = int(_avail / 8. / 3.6977678498329843e-09 / _T)
    _npx = int(np.minimum(_npx, np.prod(cnmf_model.dims) // n_processes))
    cnmf_model.params.set('preprocess', {'n_pixels_per_process': _npx})
    cnmf_model.params.set('spatial', {'n_pixels_per_process': _npx})
_Yr = np.transpose(np.reshape(images, (_T, -1), order='F'))   # same Yr fit() builds internally
_Yr = cnmf_model.preprocess(_Yr)        # per-pixel noise (sn), needed by update_temporal
cnmf_model.update_temporal(_Yr)         # solve traces C for the FIXED CellPose footprints A
cnmf_refit = cnmf_model                 # alias so the comparison cell below reads this result
if debug_tracker is not None:
    _e = cnmf_model.estimates
    debug_tracker.save_stage('final', dims=cnmf_model.dims, A=_e.A, C=_e.C,
                             S=getattr(_e, 'S', None), b=_e.b, f=_e.f, YrA=getattr(_e, 'YrA', None))
print(f'Done — {cnmf_model.estimates.A.shape[1]} cells, each with a trace. A frozen = CellPose seeds.')

In [ ]:
cnmf_model.estimates.plot_contours_nb(img=max_projection_orig)

### (Optional) Refit — OFF by default (re-grows the frozen footprints)

In [ ]:
# (Optional) Refit — OFF by default.
# Refit re-runs CNMF's spatial+temporal update on the whole FOV. For the FROZEN hybrid this
# GROWS the footprints back into merges (we measured merges 19->31), so it's OFF. Turn on only
# to reproduce that experiment.
RUN_REFIT = False
if RUN_REFIT:
    _set_debug('refit', DEBUG_REFIT)
    cnmf_refit = cnmf_model.refit(images, dview=cluster)
    print('Refit done — footprints re-optimized on the full FOV (expect the separation to degrade).')
else:
    cnmf_refit = cnmf_model
    print('Refit OFF — using the frozen model (footprints stay = CellPose seeds).')

### (Optional) Component evaluation — the junk-reduction step · OFF by default

In [ ]:
# (Optional) Component evaluation — OFF by default.
# This is the JUNK-REDUCTION step: CNMF drops components with low SNR / poor spatial correlation.
# On this data it also removes REAL faint cells (coverage 188->137 strict, 188->177 relaxed) for
# little junk gain, so it's OFF. Turn on + tune the thresholds to experiment.
RUN_COMPONENT_EVAL = False
if RUN_COMPONENT_EVAL:
    min_SNR, rval_thr = 1.0, 0.7          # relaxed; raise to drop more junk (also loses more real cells)
    cnmf_refit.params.set('quality', {'min_SNR': min_SNR, 'rval_thr': rval_thr, 'use_cnn': False})
    cnmf_refit.estimates.evaluate_components(images, cnmf_refit.params, dview=cluster)
    print('eval: kept', len(cnmf_refit.estimates.idx_components),
          '| rejected', len(cnmf_refit.estimates.idx_components_bad))
    cnmf_refit.estimates.select_components(use_object=True)
    print('components after eval:', cnmf_refit.estimates.A.shape[1])
    if debug_tracker is not None:                     # save 'evaluated' stage for the scorer/viewer
        _set_debug('refit', True)
        _e = cnmf_refit.estimates
        debug_tracker.save_stage('evaluated', dims=dims, A=_e.A, C=_e.C, S=_e.S, b=_e.b, f=_e.f, YrA=_e.YrA)
else:
    print('Component evaluation OFF — keeping all CellPose cells (score fit/final).')

### Compare against the manual ground-truth annotation

Overlays the CNMF component contours on the manually-tagged mask (auto-selected for this recording) next to the real image. **False merge** = one contour spanning two marked cells; **false split** = several contours on one marked cell. Re-run after each `merge_thr` change and screenshot for the test log.

In [ ]:
%matplotlib inline
# === Compare CNMF result vs the manual ground-truth annotation (by eye) ===
# Left:  your manual tags only (ground truth).
# Right: overlay — red CNMF outlines on your tags.
#   False MERGE = one red outline around TWO+ tagged blobs.  False SPLIT = TWO+ outlines on one blob.
# NOTE: this is the quick by-eye overlay; ground_truth_scorer.py produces the richer color-coded scoring.
import os
import tifffile
import matplotlib.pyplot as plt
from caiman.utils.visualization import plot_contours

annotation = tifffile.imread(annotation_path).astype('float32')
print(f'Loaded annotation: {annotation_path}  shape={annotation.shape}')

# pick the result to compare (refit if present, else the fit)
_est = cnmf_refit.estimates if 'cnmf_refit' in globals() else cnmf_model.estimates
_n = _est.A.shape[1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# left) your manual tags only (ground truth)
axes[0].imshow(annotation, cmap='gray')
axes[0].set_title('Your manual tags only (ground truth)')

# right) overlay: red CNMF outlines on your tags
plt.sca(axes[1])
plot_contours(_est.A, annotation, thr=0.9, display_numbers=False, colors='red', cmap='gray')
axes[1].set_title(f'Overlay: red = CellPose-seeded CNMF ({_n} comp.), light = your tags  (merge_thr={merge_thr})')

plt.tight_layout()

# Save into TEST PLAN 3's subfolder with a UNIQUE, traceable name.
# Tag = the debug run id (ties the figure back to data/results/debug_outputs/run_<id>/);
# falls back to the key params if debug is off.
_out_dir = os.path.join('..', 'data', 'results', 'comparisons', 'test-plan-3')
os.makedirs(_out_dir, exist_ok=True)
_run_id = getattr(debug_tracker, 'run_id', None) if ('debug_tracker' in globals() and debug_tracker is not None) else None
_tag = _run_id if _run_id else f'seeded_mthr{merge_thr}_n{_n}'
_out_png = os.path.join(_out_dir, f'notebook-overlay_seeded_{_tag}.png')
fig.savefig(_out_png, dpi=130, bbox_inches='tight')
print(f'Saved comparison figure to: {os.path.abspath(_out_png)}')

plt.show()